# 8. Deep Learning Architectures

## 8.1 CNN

### 8.1.1 Introduction

For images, if we simply apply fully connected networks, it will produce massive trainable parameters. Thus, CNN is introduced. By using convolutional filters and spatial structure of images, it can reduce the amount of parameters.

There are three important characteristics about CNN:

1. Local connectivity: one output element only depends on certain areas of the input;
2. Weight sharing: the same filter uses the same parameters throughout different spatial locations;
3. Feature hierarchy: multiple layers of CNN can build up different hierarchy of features, like Edges - Textures - Object.

### 8.1.2 Convolution Calculation

In deep-learning frameworks, convolution most often means Cross-Correlation, that is to do element-wise multiplication and sum on kernels and specific areas. For Example:


In [ ]:
import numpy as np

X = np.array(
    [
        [1, 2, 3],
        [4, 5, 6],
        [7, 8, 9],
    ]
)

K = np.array(
    [
        [1, 0],
        [0, -1],
    ]
)

When using Stride=1, Padding=0, the output of the upper-left corner will be:

$$
1(1) + 2(0) + 4(0) + 5(-1) = -4
$$

For a 2D Convolution:

$$
H_{out} = \lfloor \frac{H+2P-K_h}{S} \rfloor + 1
$$

$$
W_{out} = \lfloor \frac{W+2P-K_w}{S} \rfloor +1
$$

in which:

- $H$, $W$ is the height and width of input;
- $K_h$, $K_w$ is the height and width of kernel;
- $P$ is the padding of each side;
- $S$ is the stride.


### 8.1.3 Multiple Filters & Channels

In reality, the input often contains multiple channels:


In [ ]:
import tensorflow as tf

tf.keras.layers.Conv2D(
    filters=16,
    kernel_size=(3, 3),
    padding="same",
    activation="relu",
)

# If the input is (32,32,3), the output will be (32,32,16)

Notice that the number of output channels is determined by the number of `filters`, instead of `kernel_size`.


### 8.1.4 Pooling Layers

Pooling is used in spatial downsampling for feature maps. There are several types of pooling:

- Max Pooling: Select the maximum value of the cell;
- Average Pooling: Calculate the mean value of the cell;
- Global Average Pooling: Calculate the mean value of the whole spatial map in one channel.

Meanwhile, no parameter is trained in pooling.


### 8.1.5 LeNet & ResNet

LeNet is a classic CNN structure, which has a fixed order of layers:

$$
Convolution \rightarrow Pooling \rightarrow Convolution \rightarrow Pooling \rightarrow Convolution \rightarrow Fullyconnected \rightarrow Output
$$

While deepening the networks, problems may appear in the model training process, such as Optimization Difficulties, and Gradient Propagation. ResNet introduces Residual Connection (Skip Connection):

$$
y = f(x) + x
$$


### 8.1.6 Keras CNN Implementation

Different from the dense-only network before, we keep the 2D spatial structure of images this time:


In [ ]:
import numpy as np
import tensorflow as tf

# Load data
(X_train, y_train), (X_test, y_test) = tf.keras.datasets.fashion_mnist.load_data()

# Normalize and add channel dimension
X_train = X_train.astype("float32") / 255.0
X_test = X_test.astype("float32") / 255.0

X_train = X_train[..., np.newaxis]
X_test = X_test[..., np.newaxis]

print(X_train.shape)  # (60000, 28, 28, 1)

# Build CNN
model = tf.keras.Sequential(
    [
        tf.keras.layers.Input(
            shape=(28, 28, 1),
        ),  # shape = (28,28,1)
        tf.keras.layers.Conv2D(
            16,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
        ),  # shape = (28,28,16)
        tf.keras.layers.MaxPooling2D(
            pool_size=(2, 2),
        ),  # shape = (14,14,16)
        tf.keras.layers.Conv2D(
            32,
            kernel_size=(3, 3),
            padding="same",
            activation="relu",
        ),  # shape = (14,14,32)
        tf.keras.layers.MaxPooling2D(
            pool_size=(2, 2),
        ),  # shape = (7,7,32)
        tf.keras.layers.Flatten(),  # shape = (1568,)
        tf.keras.layers.Dense(
            64,
            activation="relu",
        ),  # shape = (64,)
        tf.keras.layers.Dense(
            10,
            activation="softmax",
        ),  # shape = (10,)
    ]
)

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

history = model.fit(
    X_train,
    y_train,
    epochs=3,
    batch_size=64,
    validation_split=0.1,
)

test_loss, test_accuracy = model.evaluate(
    X_test,
    y_test,
    verbose=0,
)

print("Test Accuracy:", test_accuracy)

in which the shape of an image is defined as:

$$
(Height, Width, Channels) = (H,W,C)
$$

Channels is the number of features of certain spatial location. For example, in a B-W image, `c=1`; In an RGB image, `c=3`. However, the number of channels is not restricted. In CNN, it can also represent the number of features.

Furtheremore, `padding="same"` auto sets the padding to ensure the shape of the output is the same as input.


### 8.1.7 Trainable Parameters

The Parameter Count of a Conv2D layer is defined as:

$$
(K_hK_wC_{in} + 1) \times C_{out}
$$
